In [1]:
import numpy as np
import pandas as pd
import requests

In [2]:
BASE_URL = "https://leaderboard-project1-runai-innovation-thibaut.inference.compute.datascience.ch/api/v1"
TEMPLATE_PATH = "./submission.csv"
TARGETS = ["vcd", "tcd", "substrate", "byproduct"]


def get_leaderboard() -> pd.DataFrame:
    leaderboard = requests.get(f"{BASE_URL}/leaderboard").json()
    return pd.json_normalize(leaderboard["entries"])


def submit(df: pd.DataFrame, user: str, token: str = "") -> dict:
    submission = {"predictions": df.to_dict(orient="records")}
    response = requests.post(f"{BASE_URL}/submit", json=submission, auth=(user, token))
    if (response.status_code // 100) != 2:
        raise RuntimeError(f"Unexpected error (HTTP {response.status_code}): {response.text}")
    return response.json()["metrics"]

In [3]:
get_leaderboard()

  user_name  rank  attempt                   timestamp  metrics.vcd_mape  \
0    Team 7     1       11  2026-11-22T11:33:02.194610             6.627   
1   Team 04     2       16  2026-11-22T10:56:05.255107             6.747   
2   Team 15     3        8  2026-11-22T08:03:13.449316             6.609   
3   Team 01     4        8  2026-11-22T11:59:10.489944             6.526   
4   Team 23     5       31  2026-11-22T11:44:57.671046             6.534   
5    Team 6     6       21  2026-11-22T12:00:16.957461             6.794   
6   Team 10     7      114  2026-11-22T11:24:58.248494             7.164   
7   Team 18     8       18  2026-11-22T11:52:17.950964             7.026   
8  TEMPLATE     9       69  2026-11-21T14:24:40.616352             8.037   

   metrics.tcd_mape  metrics.substrate_mape  metrics.byproduct_mape  \
0             6.569                   6.326                   7.314   
1             6.550                   6.633                   6.969   
2             6.463       

In [4]:
template = pd.read_csv(TEMPLATE_PATH, dtype={"batch_id": str})
# Change TEMPLATE with your team name as "Team XX", for example "Team 21" 
metrics = submit(template, user="Team 10")
metrics

{'vcd_mape': 7.207,
 'tcd_mape': 7.183,
 'substrate_mape': 8.285,
 'byproduct_mape': 8.307,
 'mean_mape': 7.745}